# Qwen3.8-27B-Uncensored (abliterated) — API OpenAI-compatible

Roda no **Kaggle** (2x T4, ~30 GB VRAM) ou **Colab** (T4, ~15 GB) e expõe uma API
compatível com OpenAI via **Cloudflare Tunnel** (grátis, sem conta).

**Como usar**
- **Primeira vez / sessão nova**: rode a **célula 2 (Baixar)** — compila o llama.cpp e baixa o modelo.
- **Depois**: rode a **célula 1 (Rodar)** — sobe o modelo com contexto grande + túnel + teste.

Kaggle: `Settings > Accelerator = GPU T4 x2` e `Internet = On`. Colab: `Runtime > T4 GPU`.

**Avisos**
- A URL `*.trycloudflare.com` morre quando a sessão cair. Rode a célula 1 de novo.
- O modelo é abliterated (recusa removida). Use por sua conta e risco.

In [ ]:
#@title 1) RODAR: modelo (contexto grande) + servidor + tunel + teste { display-mode: "form" }
import os, subprocess, time, urllib.request, urllib.error, re, json

IS_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if IS_KAGGLE else "/content"
PORT = 8080
CTX_CANDIDATES = [262144, 131072, 65536, 32768]   # tenta o maior primeiro
API_KEY = globals().get("API_KEY") or ("sk-local-" + os.urandom(8).hex())

gpu_raw = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, text=True, capture_output=True).stdout.strip()
gpus = []
for line in gpu_raw.splitlines():
    p = [x.strip() for x in line.split(",")]
    if len(p) >= 2:
        gpus.append({"name": p[0], "vram_gb": int(p[1].split()[0]) / 1024})
assert gpus, "Sem GPU NVIDIA. Ative o acelerador (T4) e reinicie."
TOTAL_VRAM = sum(g["vram_gb"] for g in gpus)
if TOTAL_VRAM >= 44:   QUANT = "Q6_K"
elif TOTAL_VRAM >= 24: QUANT = "Q4_K_M"
elif TOTAL_VRAM >= 14: QUANT = "Q3_K_S"
else:                  QUANT = "IQ2_M"

model_path = os.path.join(WORK, "models", f"Qwen3.8-27B-Uncensored-{QUANT}.gguf")
BIN = os.path.join(WORK, "llama.cpp", "build", "bin", "llama-server")
assert os.path.exists(model_path), f"Modelo nao encontrado: {model_path} (rode a celula 2 primeiro)"
assert os.path.exists(BIN), "llama-server nao compilado (rode a celula 2 primeiro)"
print(f"GPUs: {len(gpus)} | VRAM: {TOTAL_VRAM:.1f} GB | quant: {QUANT} | API_KEY: {API_KEY}")

BINDIR = os.path.join(WORK, "llama.cpp", "build", "bin")
env = dict(os.environ)
env["LD_LIBRARY_PATH"] = BINDIR + ":/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")
SERVER_LOG = os.path.join(WORK, "llama-server.log")

# espera o servidor responder de verdade (503 = ainda carregando o modelo)
def server_ready(proc, timeout=900):
    start = time.time()
    while time.time() - start < timeout:
        if proc.poll() is not None:
            return False
        try:
            b = json.dumps({"model": "local", "messages": [{"role": "user", "content": "ping"}], "max_tokens": 1}).encode()
            r = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", data=b,
                headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
            urllib.request.urlopen(r, timeout=15)
            return True
        except urllib.error.HTTPError as e:
            if e.code == 503:
                time.sleep(5); continue
            return False
        except Exception:
            time.sleep(5)
    return False

CTX = None
for cand in CTX_CANDIDATES:
    os.system("pkill -f llama-server; sleep 2")
    cmd = [BIN, "-m", model_path, "--host", "0.0.0.0", "--port", str(PORT),
           "-ngl", "99", "-c", str(cand), "--parallel", "1", "--jinja", "--api-key", API_KEY,
           "--cache-type-k", "q8_0", "--cache-type-v", "q8_0", "-fa", "on", "-rea", "off"]
    srv = subprocess.Popen(cmd, stdout=open(SERVER_LOG, "w"), stderr=subprocess.STDOUT, env=env)
    print(f"Tentando contexto {cand}...")
    if server_ready(srv):
        CTX = cand
        print(f"Servidor no ar! contexto = {CTX}")
        break
    print(f"Falhou com {cand}, tentando menor. Ultimas linhas do log:")
    print(open(SERVER_LOG).read()[-1200:])
if CTX is None:
    raise SystemExit("Nao consegui subir o servidor em nenhum contexto.")

print("--- offload (procure 'CUDA' e 'offloaded ... layers') ---")
for line in open(SERVER_LOG).read().splitlines():
    if any(k in line for k in ("CUDA", "offloaded", "VRAM", "backend")):
        print(line)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.used --format=csv,noheader",
                     shell=True, text=True, capture_output=True).stdout)

# --- tunel ---
CF = os.path.join(WORK, "cloudflared")
if not os.path.exists(CF):
    os.system(f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF} && chmod +x {CF}")
CLOG = os.path.join(WORK, "cloudflared.log")
os.system("pkill -f cloudflared; sleep 1")
subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                 stdout=open(CLOG, "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for i in range(90):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(CLOG).read())
    if m: PUBLIC_URL = m.group(0); break
print("PUBLIC_URL =", PUBLIC_URL)
assert PUBLIC_URL, "Nao criei o tunel"

# --- teste + info ---
payload = json.dumps({"model": "local",
    "messages": [{"role": "user", "content": "Responda apenas: OK"}],
    "max_tokens": 32}).encode()
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", data=payload,
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
print("Resposta:", json.loads(urllib.request.urlopen(req, timeout=300).read().decode())["choices"][0]["message"]["content"])
print(f"\nBase URL : {PUBLIC_URL}/v1\nAPI key  : {API_KEY}\nContexto : {CTX}")

In [ ]:
#@title 2) BAIXAR (primeira vez): compilar o llama.cpp e baixar o modelo { display-mode: "form" }
import os, sys, subprocess

def sh(cmd, timeout=None):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, text=True, timeout=timeout)

IS_KAGGLE = os.path.exists("/kaggle")
gpu_raw = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader",
    shell=True, text=True, capture_output=True).stdout.strip()
gpus = []
for line in gpu_raw.splitlines():
    p = [x.strip() for x in line.split(",")]
    if len(p) >= 3:
        gpus.append({"name": p[0], "vram_gb": int(p[1].split()[0]) / 1024, "cap": p[2].replace(".", "")})
TOTAL_VRAM = sum(g["vram_gb"] for g in gpus)
assert gpus, "Sem GPU NVIDIA. Ative o acelerador (T4) e reinicie."
WORK = "/kaggle/working" if IS_KAGGLE else "/content"
if TOTAL_VRAM >= 44:
    QUANT = "Q6_K"
elif TOTAL_VRAM >= 24:
    QUANT = "Q4_K_M"
elif TOTAL_VRAM >= 14:
    QUANT = "Q3_K_S"
else:
    QUANT = "IQ2_M"
MODEL_REPOS = [
    "orcarouter/Qwen3.8-27B-Uncensored-GGUF",        # oficial (gated)
    "JonathanColetti/Qwen3.8-27B-Uncensored-GGUF",  # mirror livre
    "mradermacher/Qwen3.8-27B-Uncensored-GGUF",      # 2o mirror livre
]
print(f"GPUs: {len(gpus)} | VRAM: {TOTAL_VRAM:.1f} GB | quant: {QUANT}")

# --- compilar (pula se o binario ja existe) ---
arch = gpus[0]["cap"]
BIN = os.path.join(WORK, "llama.cpp", "build", "bin", "llama-server")
if os.path.exists(BIN):
    print("llama-server ja compilado, pulando build:", BIN)
else:
    os.chdir(WORK)
    sh("apt-get -qq update && apt-get -qq install -y build-essential cmake git libcurl4-openssl-dev >/dev/null 2>&1")
    # garante libcuda.so (driver) para o CMake achar CUDA::cuda_driver
    sh("mkdir -p /usr/local/cuda/lib64 && ln -sf $(find / -name 'libcuda.so.1' 2>/dev/null | head -1) /usr/local/cuda/lib64/libcuda.so")
    if not os.path.isdir("llama.cpp"):
        sh("git clone --depth 1 https://github.com/ggml-org/llama.cpp", timeout=600)
    os.chdir("llama.cpp")
    sh(f"cmake -B build -DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES={arch} -DLLAMA_BUILD_UI=OFF -DLLAMA_USE_PREBUILT_UI=OFF", timeout=600)
    sh("cmake --build build --config Release -j$(nproc) -t llama-server", timeout=5400)
    print("Binario:", BIN)

# --- baixar o modelo (pula se ja existe) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub[hf_transfer]"])
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN carregado do Kaggle Secrets")
except Exception:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN carregado do Colab Secrets")
    except Exception:
        print("Sem HF_TOKEN (token nao obrigatorio)")

from huggingface_hub import hf_hub_download
MODELS = os.path.join(WORK, "models")
os.makedirs(MODELS, exist_ok=True)
model_file = f"Qwen3.8-27B-Uncensored-{QUANT}.gguf"
model_path = os.path.join(MODELS, model_file)
if os.path.exists(model_path) and os.path.getsize(model_path) > 5e9:
    print("Modelo ja baixado:", model_path)
else:
    model_path = None
    for repo in MODEL_REPOS:
        try:
            model_path = hf_hub_download(repo, model_file, local_dir=MODELS)
            print("Baixado de:", repo); break
        except Exception as e:
            print(f"Falhou {repo} ({type(e).__name__}) - tentando o proximo...")
    assert model_path, "Nao consegui baixar de nenhum mirror."
print("Modelo:", model_path, f"({os.path.getsize(model_path)/1e9:.2f} GB)")

## Dicas

- **Contexto**: a célula 1 tenta 64k e cai pra 32k/16k se faltar VRAM. No fim ela imprime `Contexto : ...`.
  Deixe o `limit.context` no `opencode.jsonc` igual a esse valor (senão o OpenCode compacta cedo ou estoura).
- A célula 1 é **re-executável** (mata servidor/túnel antigos antes de subir).
- A URL `*.trycloudflare.com` muda a cada execução — atualize o `baseURL` no `opencode.jsonc`.
- Quer o **GLM-5.3-Flash-Uncensored** (320B)? Não cabe em GPU grátis — use o `obsidian/Qwen3.8-27B`
  na OrcaRouter com créditos grátis de `orcarouter.ai/offers`.